<a href="https://colab.research.google.com/github/silvanhess/parldebates_analysis/blob/main/topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.8 MB/s eta 0:00:00


In [5]:
import pandas as pd
from bertopic import BERTopic
import hdbscan
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [1]:
from google.colab import files
uploaded = files.upload()

Saving transcripts_climate_for_topic_modeling.csv to transcripts_climate_for_topic_modeling.csv


In [6]:
# 1. Read in CSV
df = pd.read_csv('transcripts_climate_for_topic_modeling.csv')
text_column = 'transcript_text'

# Extract the text data
documents = df[text_column].tolist()

In [8]:
# 2. Initialize BERTopic model

# For 37,000 documents with ~200 words each
umap_model = UMAP(
    n_neighbors=15,  # Keep default, don't over-smooth
    n_components=10,  # Increase to preserve more variance
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=500,  # Much larger - force it to split the mega-cluster
    min_samples=50,  # Higher to avoid merging everything
    metric='euclidean',
    cluster_selection_method='leaf',  # Try 'leaf' instead of 'eom' for more splits
    prediction_data=True
)

# Configure CountVectorizer to filter rare and very common words
vectorizer_model = CountVectorizer(
    min_df=2,  # Ignore words that appear in fewer than 2 documents
    max_df=0.95,  # Ignore words that appear in more than 95% of documents
    ngram_range=(1, 2)  # Include both single words and bigrams
)

topic_model = BERTopic(
    language="multilingual",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,  # Filter rare/common words
    nr_topics="auto",  # Allows automatic merging if too many topics
    verbose=True
)

In [ ]:
# 3. Fit the model and transform documents
topics, probs = topic_model.fit_transform(documents)

2026-01-14 12:19:38,335 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1174 [00:00<?, ?it/s]

2026-01-14 12:21:31,122 - BERTopic - Embedding - Completed ✓
2026-01-14 12:21:31,123 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-14 12:22:35,311 - BERTopic - Dimensionality - Completed ✓
2026-01-14 12:22:35,315 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-14 12:22:45,541 - BERTopic - Cluster - Completed ✓
2026-01-14 12:22:45,542 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-01-14 12:23:06,822 - BERTopic - Representation - Completed ✓
2026-01-14 12:23:06,848 - BERTopic - Topic reduction - Reducing number of topics
2026-01-14 12:23:06,866 - BERTopic - Representation - Fine-tuning topics using representation models.


In [ ]:
# Add topics back to the dataframe
df['topic'] = topics
df['topic_probability'] = probs

# Display basic information
print(f"\nNumber of topics found: {len(set(topics)) - 1}")  # -1 for outlier topic
print("\nTopic distribution:")
print(df['topic'].value_counts().head(10))

# Get topic information
topic_info = topic_model.get_topic_info()
print("\nTopic Information:")
print(topic_info.head(10))

In [ ]:
# Save results
df.to_csv('Data/documents_with_topics.csv', index=False)
topic_info.to_csv('Data/topic_info.csv', index=False)